# ADL 원시 데이터 적재

`sample/`의 데이터바우처 엑셀 샘플(응급/사망 발생 ADL) 2개를 읽어 `adl_raw_records`
테이블에 적재한다.

**실행 순서**
1. 먼저 `adl_raw_reset.ipynb`를 실행해 테이블을 비운다.
2. 이 노트북을 위에서 아래로 전체 실행한다.

**참고**: 엑셀 5개 컬럼은 hex 문자열이라 정수 리스트로 디코딩해 PostgreSQL 배열
(`int[]` / `double precision[]`) 컬럼에 저장한다.

In [1]:
import math
import os
import sys
from pathlib import Path

# 작업 디렉토리를 프로젝트 루트로 이동(.env.local·app import 해석용, 폴더 생성 아님).
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd

SAMPLE_DIR = ROOT / "sample"
EMERGENCY_FILE = SAMPLE_DIR / "05-1 데이터바우처 지원사업 데이터 샘플(응급상황 발생 ADL)_리본케어.xlsx"
DEATH_FILE = SAMPLE_DIR / "05-2 데이터바우처 지원사업 데이터 샘플(사망 발생 ADL)_리본케어.xlsx"

print(f"프로젝트 루트: {ROOT}")
print(f"엑셀 존재: 응급={EMERGENCY_FILE.exists()}, 사망={DEATH_FILE.exists()}")

프로젝트 루트: C:\Users\kdwkd\documents\project\salpyeobom\salpyeobom-backend
엑셀 존재: 응급=True, 사망=True


## 1. hex 디코딩

엑셀의 5개 컬럼(`place_code_1_list`, `AIX_1_list`, `AIX_h_list`, `sleep_depth_1_list`,
`outgoing_1_list`)은 리스트가 아니라 **hex 문자열**이다 (`0000…`, `FFFE…`, `0x0404…`).
아래 함수가 이를 정수 리스트로 디코딩한다.

- 값 1개 = **1바이트**: `place_code` · `sleep_depth` · `outgoing` (분 단위 1440개)
- 값 1개 = **2바이트 big-endian**: `AIX_1`(1440개) · `AIX_h`(시 단위 24개)
- `temp/humi/illu`는 hex가 아니라 `temp_00`~`temp_23` 개별 컬럼 → 별도 함수로 수집

In [2]:
def hex_to_int_list(value: object, width: int = 1) -> list[int] | None:
    """hex 문자열을 정수 리스트로 디코딩한다.

    width : 값 1개당 바이트 수. 1 = 1바이트, 2 = 2바이트 big-endian.
    hex 형식이 아니면(예: 사망 파일의 정수 0) None 을 반환한다.
    """
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return None
    s = str(value).strip().lower()
    if s.startswith("0x"):
        s = s[2:]
    if not s or len(s) % (2 * width) != 0:
        return None
    if any(c not in "0123456789abcdef" for c in s):
        return None
    raw = bytes.fromhex(s)
    if width == 1:
        return list(raw)
    return [int.from_bytes(raw[i:i + width], "big") for i in range(0, len(raw), width)]

In [3]:
def nan_to_none(v: object) -> object:
    if v is None or (isinstance(v, float) and math.isnan(v)):
        return None
    return v


def parse_date(v: object):
    """YYYYMMDD 정수 또는 날짜 문자열을 date 로 파싱."""
    if nan_to_none(v) is None:
        return None
    try:
        s = str(int(float(str(v))))
        if len(s) == 8:
            return pd.to_datetime(s, format="%Y%m%d").date()
    except (ValueError, OverflowError):
        pass
    parsed = pd.to_datetime(v, errors="coerce")
    return None if pd.isna(parsed) else parsed.date()


def parse_int(v: object) -> int | None:
    if nan_to_none(v) is None:
        return None
    try:
        return int(float(str(v)))
    except (ValueError, TypeError):
        return None


def parse_float(v: object) -> float | None:
    if nan_to_none(v) is None:
        return None
    try:
        return float(str(v))
    except (ValueError, TypeError):
        return None


def parse_str(v: object, max_len: int | None = None) -> str | None:
    if nan_to_none(v) is None:
        return None
    s = str(v).strip()
    if not s or s.lower() == "nan":
        return None
    return s[:max_len] if max_len else s


def parse_id(v: object, max_len: int) -> str:
    """ID 컬럼 전용 정규화. pandas가 NaN 섞인 컬럼을 float64로 읽어 '661.0'으로
    저장되는 문제를 막기 위해, 정수로 변환 가능하면 정수 문자열로 반환한다."""
    n = parse_int(v)
    if n is not None:
        return str(n)[:max_len]
    return parse_str(v, max_len) or ""


def extract_hourly(row: pd.Series, prefix: str) -> list | None:
    """temp_00 ~ temp_23 같은 시간별 컬럼 24개를 리스트로 모은다."""
    vals = [parse_float(row.get(f"{prefix}_{h:02d}")) for h in range(24)]
    return None if all(v is None for v in vals) else vals

In [4]:
def build_record(row: pd.Series, source_type: str) -> dict:
    """엑셀 한 행을 AdlRawRecord 필드 dict 로 변환한다."""
    return {
        "source_type": source_type,
        # 기본 정보
        "care_recipient_id": parse_id(row.get("care_recipient_id"), 32),
        "age": parse_int(row.get("age")),
        "sex": parse_str(row.get("sex"), 1),
        "alone": parse_str(row.get("alone"), 1),
        "vision": parse_str(row.get("vision"), 16),
        "hearing": parse_str(row.get("hearing"), 16),
        "dosage": parse_str(row.get("dosage"), 16),
        "district": parse_str(row.get("district"), 64),
        "house_structure": parse_str(row.get("house_structure"), 16),
        "room_no": parse_int(row.get("room_no")),
        "bath_location": parse_str(row.get("bath_location"), 16),
        # 이벤트 정보
        "lifeog_date": parse_date(row.get("lifeog_date")),
        "emergency_date": parse_date(row.get("emergency_date")),
        "emergency_record": parse_str(row.get("emergency_record")),
        "occurrence_place": parse_str(row.get("occurrence_place"), 32),
        "on_site": parse_str(row.get("on_site"), 16),
        "hospital_transfer": parse_str(row.get("hospital_transfer"), 16),
        "hospital_treatment": parse_str(row.get("hospital_treatment"), 16),
        "death_date": parse_date(row.get("death_date")),
        "death_record": parse_str(row.get("death_record")),
        # 분석 데이터 — hex 문자열을 정수 리스트로 디코딩
        "place_code_1_list": hex_to_int_list(row.get("place_code_1_list"), width=1),
        "aix_1_list": hex_to_int_list(row.get("AIX_1_list"), width=2),
        "aix_h_list": hex_to_int_list(row.get("AIX_h_list"), width=2),
        "aix_d": parse_float(row.get("AIX_d")),
        "aix_1_eq_0_repeat_count": parse_int(row.get("AIX_1_eq_0_repeat_count")),
        "total_aix_sum": parse_float(row.get("total_aix_sum")),
        "total_aix_inc_ratio": parse_float(row.get("total_aix_inc_ratio")),
        "night_aix_ratio": parse_float(row.get("night_aix_ratio")),
        "total_age_aix_ratio": parse_float(row.get("total_age_aix_ratio")),
        # 수면
        "sleep_depth_1_list": hex_to_int_list(row.get("sleep_depth_1_list"), width=1),
        "sleep_start_time_d": parse_str(row.get("sleep_start_time_d"), 8),
        "sleep_end_time_d": parse_str(row.get("sleep_end_time_d"), 8),
        "total_sleep_period": parse_float(row.get("total_sleep_period")),
        "total_sleep_aix_ratio": parse_float(row.get("total_sleep_aix_ratio")),
        # 목욕
        "bath_count_d": parse_int(row.get("bath_count_d")),
        "bath_time_d": parse_float(row.get("bath_time_d")),
        "bath_nomove_time": parse_float(row.get("bath_nomove_time")),
        "bath_count_in_sleep": parse_int(row.get("bath_count_in_sleep")),
        "bath_time_per_count": parse_float(row.get("bath_time_per_count")),
        "total_bath_average_count": parse_float(row.get("total_bath_average_count")),
        # 외출
        "outgoing_1_list": hex_to_int_list(row.get("outgoing_1_list"), width=1),
        "outgoing_count_d": parse_int(row.get("outgoing_count_d")),
        "outgoing_time_d": parse_float(row.get("outgoing_time_d")),
        "outgoing_late_night_count_d": parse_int(row.get("outgoing_late_night_count_d")),
        "outgoing_late_night_time_d": parse_float(row.get("outgoing_late_night_time_d")),
        "last_outgoing_time": parse_str(row.get("last_outgoing_time"), 16),
        "total_outgoing_average_time": parse_float(row.get("total_outgoing_average_time")),
        "total_outgoing_average_count": parse_float(row.get("total_outgoing_average_count")),
        # 시간별 환경 센서 — temp_00~temp_23 등 24개 컬럼 수집
        "temp_list": extract_hourly(row, "temp"),
        "humi_list": extract_hourly(row, "humi"),
        "illu_list": extract_hourly(row, "illu"),
    }

In [5]:
frames = []
for path, source_type in [(EMERGENCY_FILE, "응급"), (DEATH_FILE, "사망")]:
    df = pd.read_excel(path, sheet_name=0)
    # care_recipient_id 는 엑셀 병합 셀이라 첫 행에만 값이 있음 → forward fill
    df["care_recipient_id"] = df["care_recipient_id"].ffill()
    df["__source_type"] = source_type
    frames.append(df)
    print(f"[{source_type}] {len(df)}행 로드: {path.name}")

records_df = pd.concat(frames, ignore_index=True)
print(f"총 {len(records_df)}행")
records_df[["care_recipient_id", "__source_type", "age"]].head()

[응급] 30행 로드: 05-1 데이터바우처 지원사업 데이터 샘플(응급상황 발생 ADL)_리본케어.xlsx
[사망] 30행 로드: 05-2 데이터바우처 지원사업 데이터 샘플(사망 발생 ADL)_리본케어.xlsx
총 60행


C:\Users\kdwkd\AppData\Local\Temp\ipykernel_28828\510867746.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["__source_type"] = source_type


,care_recipient_id,__source_type,age
0,661,응급,78
1,661,응급,78
2,661,응급,78
3,661,응급,78
4,661,응급,78


In [6]:
from tortoise import Tortoise

from app.database import TORTOISE_ORM
from app.models.adl_raw import AdlRawRecord

# 스키마는 aerich 가 관리하므로 generate_schemas() 는 호출하지 않는다.
# 노트북 재실행 시 중복 init 방지를 위해 public Tortoise.apps 로 상태 확인.
if not Tortoise.apps:
    await Tortoise.init(config=TORTOISE_ORM)
print("DB 연결 완료")

DB 연결 완료


In [7]:
from tortoise.transactions import in_transaction

existing = await AdlRawRecord.all().count()
if existing:
    raise RuntimeError(
        f"adl_raw_records 에 이미 {existing}건이 있습니다. "
        "먼저 adl_raw_reset.ipynb 를 실행해 테이블을 비우세요."
    )

records = [
    AdlRawRecord(**build_record(row, row["__source_type"]))
    for _, row in records_df.iterrows()
]

async with in_transaction():
    await AdlRawRecord.bulk_create(records)

print(f"adl_raw_records: {len(records)}건 적재 완료")

adl_raw_records: 60건 적재 완료


In [8]:
total = await AdlRawRecord.all().count()
sample = await AdlRawRecord.first()
print(f"총 {total}건 적재됨")
print(f"source_type={sample.source_type}, care_recipient_id={sample.care_recipient_id}")

# 8개 배열 컬럼이 정수/실수 리스트로 정상 적재됐는지 확인
array_cols = [
    "place_code_1_list", "aix_1_list", "aix_h_list", "sleep_depth_1_list",
    "outgoing_1_list", "temp_list", "humi_list", "illu_list",
]
for col in array_cols:
    v = getattr(sample, col)
    length = len(v) if isinstance(v, list) else None
    head = v[:8] if isinstance(v, list) else v
    print(f"  {col}: 길이={length}, 앞부분={head}")

총 60건 적재됨
source_type=응급, care_recipient_id=661
  place_code_1_list: 길이=1440, 앞부분=[0, 0, 0, 0, 0, 0, 0, 0]
  aix_1_list: 길이=1440, 앞부분=[0, 0, 0, 0, 0, 0, 0, 0]
  aix_h_list: 길이=24, 앞부분=[18, 7, 4, 0, 52, 311, 110, 13]
  sleep_depth_1_list: 길이=1440, 앞부분=[4, 4, 4, 4, 4, 4, 4, 4]
  outgoing_1_list: 길이=1440, 앞부분=[255, 254, 254, 254, 254, 254, 254, 254]
  temp_list: 길이=24, 앞부분=[30.0, 30.0, 29.8, 29.6, 29.4, 29.2, 29.1, 29.0]
  humi_list: 길이=24, 앞부분=[59.7, 59.6, 60.1, 60.4, 60.8, 61.2, 61.4, 61.8]
  illu_list: 길이=24, 앞부분=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 5.0, 10.0]


In [9]:
await Tortoise.close_connections()
print("DB 연결 종료")

DB 연결 종료
